In [153]:
import time

import requests
import os
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

load_dotenv()

account_name = os.getenv("VTEX_ACCOUNT_NAME")
app_key = os.getenv("VTEX_APP_KEY")
app_token = os.getenv("VTEX_APP_TOKEN")
headers = {
    'Accept': 'application/json',
	'Content-Type': 'application/json',
	'X-VTEX-API-AppKey': app_key,
	'X-VTEX-API-AppToken': app_token
}
BASE_URL = f"https://{account_name}.vtexcommercestable.com.br/api/catalog_system/pvt"

In [154]:
@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=2, max=10),
    retry=retry_if_exception_type(requests.exceptions.RequestException)
)
def vtex_get(path, params=None):
    """Função genérica para requisições GET na VTEX."""
    url = f"{BASE_URL}/{path}"
    time.sleep(1)
    response = requests.get(url, headers=headers, params=params, timeout=5)
    response.raise_for_status()
    return response.json()

In [155]:
# Vtex funcions
def total_ids():
    try:
        res = vtex_get("products/GetProductAndSkuIds")
        return res['range']['total']
    except (requests.exceptions.RequestException, KeyError) as e:
        print(f"Não foi possível capturar todos os Ids: {e}")
        return None

def sku_id_list(start, end):
    try:
        res = vtex_get("products/GetProductAndSkuIds", params={'_from': start, '_to': end})
        return res.get('data', {})
    except requests.exceptions.RequestException as e:
        print(f"Não foi possível capturar a lista de Ids: {e}")
        return None

def fetch_refid_using_product_id(productid):
    try:
        res = vtex_get(f"sku/stockkeepingunitByProductId/{productid}")
        return res[0]['RefId']
    except (requests.exceptions.RequestException, IndexError, KeyError) as e:
        print(f"Não foi possível capturar o Refid do product {productid}: {e}")
        return None

def fetch_sku_ids(salto, total_sku_ids):
    inicio = 0  # começa do 0
    dict_sku_ids = {}

    try:
        while inicio < total_sku_ids:
            fim = min(inicio + salto - 1, total_sku_ids)
            page = sku_id_list(inicio, fim)
            if page:
                dict_sku_ids.update(page)
            inicio += salto
    except Exception as e:
        print(f"❌ Erro ao buscar SKUs: {e}")
        return {}

    return dict_sku_ids

In [156]:
def complete_refid_list():
    refid_list = []
    total = total_ids()

    if total is None:
        print("❌ Falha ao obter o total de produtos. Encerrando processo.")
        return refid_list  # ou `return None` se preferir

    print(f"🔢 Total de produtos: {total}")
    dict_sku_ids = fetch_sku_ids(250, total)

    for key, value in dict_sku_ids.items():
        try:
            ref_id = fetch_refid_using_product_id(key)
            print(f"Salvando Product Id: {key}, Sku (Id): {value[0]}, RefId: {ref_id}")
            refid_list.append(ref_id)
        except Exception as e:
            print(f"❌ Falha ao buscar RefID para Product ID {key}: {e}")

    return refid_list

complete_refid_list()

🔢 Total de produtos: 526
Salvando Product Id: 1, Sku (Id): 1, RefId: 409312
Salvando Product Id: 2, Sku (Id): 2, RefId: 406929
Salvando Product Id: 3, Sku (Id): 3, RefId: 418297
Salvando Product Id: 4, Sku (Id): 4, RefId: 416621
Salvando Product Id: 5, Sku (Id): 5, RefId: 1936
Salvando Product Id: 6, Sku (Id): 6, RefId: 16500
Salvando Product Id: 7, Sku (Id): 7, RefId: 432657
Salvando Product Id: 8, Sku (Id): 8, RefId: 420914
Salvando Product Id: 9, Sku (Id): 9, RefId: 419961
Salvando Product Id: 10, Sku (Id): 10, RefId: 423294
Salvando Product Id: 11, Sku (Id): 11, RefId: 423300
Salvando Product Id: 12, Sku (Id): 12, RefId: 422951
Salvando Product Id: 13, Sku (Id): 13, RefId: 409944
Salvando Product Id: 14, Sku (Id): 14, RefId: 424882
Salvando Product Id: 15, Sku (Id): 15, RefId: 417073
Salvando Product Id: 16, Sku (Id): 16, RefId: 417074
Salvando Product Id: 17, Sku (Id): 17, RefId: 416619
Salvando Product Id: 18, Sku (Id): 46, RefId: 407509
Salvando Product Id: 19, Sku (Id): 27, Ref

['409312',
 '406929',
 '418297',
 '416621',
 '1936',
 '16500',
 '432657',
 '420914',
 '419961',
 '423294',
 '423300',
 '422951',
 '409944',
 '424882',
 '417073',
 '417074',
 '416619',
 '407509',
 '424347',
 '414370',
 '999999',
 '412647',
 '407506',
 '421048',
 '402944',
 '424391',
 '425457',
 '432652',
 '402943',
 '24384',
 '421044',
 '420930',
 '430134',
 '420118',
 '430332',
 '144594',
 '103217',
 '425193',
 '407226',
 '420943',
 '117273',
 '430434',
 '418384',
 '436427',
 '432654',
 '427007',
 '430010',
 '418299',
 '432611',
 '429754',
 '427566',
 '425195',
 '424912',
 '425194',
 '412022',
 '412744',
 '420917',
 '139969',
 '415298',
 '417749',
 '411490',
 '412750',
 '400527',
 '411488',
 '411489',
 '124838',
 '417930',
 '124830',
 '400528',
 '402617',
 '400529',
 '419625',
 '417422',
 '431650',
 '423004',
 '415863',
 '420931',
 '417431',
 '417429',
 '417304',
 '417661',
 '437536',
 '437537',
 '436857',
 '430779',
 '430780',
 '413101',
 '417666',
 '438618',
 '430794',
 '430795',
 '4